# Phase 2 ? B2 Uniform Experience Replay (Kaggle)

Runs B2 on the frozen five-language XNLI protocol using the Phase 1 candidate-label prompt and shared LoRA trainer. It runs all five fixed stream orders (one training seed each), evaluates every language before training and after each stage, saves stage adapters plus result ledgers, and writes a validated result JSON per completed order. B2 uses a 125-example uniform reservoir. Before each later language stage, it samples one replay example with replacement per current example from previously seen languages; after the stage, that language's examples enter the reservoir. The first stage has no replay. This 1:1 ratio is explicitly documented because the research plan leaves it unspecified; replay increases total example exposure and compute relative to B0.

## Before running
1. Enable a Kaggle GPU and Internet.
2. Accept the gated model terms for `meta-llama/Llama-3.2-1B` and add a Kaggle secret named `HF_TOKEN`.
3. This GitHub repository is public, so no GitHub token/API key is needed to clone it. The URL currently redirects to a repository name ending in a period; keep the exact `REPO_URL` below.
4. Add optional `WANDB_API_KEY` only if you enable W&B.
5. The B2 runner must be committed and pushed to the public GitHub repository before Kaggle can clone it. Local-only changes are not included in Kaggle's clone.
6. Run all cells, then use **Save Version** to persist `/kaggle/working/phase2_artifacts` as notebook output. A run interrupted during training preserves its attempt outputs; rerun with the same artifact root to allocate a fresh attempt, while completed orders are validated and skipped.

This notebook runs all five B2 orders; it does not offer a partial-run mode. Completed order results are validated and skipped on rerun, so rerunning safely resumes only missing orders. At the end it verifies all five result records and writes `B2_summary.json`. B2 result metadata records the replay ratio, sampling seed, buffer size by stage, replay language counts, source revision, and runtime packages.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Mani-S-49/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
repo_revision = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip()
print("Cloned repository revision:", repo_revision)

requirements_file = REPO_ROOT / "phase2" / "requirements-phase2.txt"
if not requirements_file.exists():
    raise FileNotFoundError(f"Requirements file not found: {requirements_file}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)], check=True)
sys.path.insert(0, str(REPO_ROOT))

runner_path = REPO_ROOT / "phase2" / "run_baseline.py"
if not runner_path.is_file():
    raise FileNotFoundError(f"Missing B2 runner: {runner_path}")

help_result = subprocess.run(
    [sys.executable, "-m", "phase2.run_baseline", "--help"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(help_result.stdout)
print(help_result.stderr)
if help_result.returncode != 0:
    raise RuntimeError("phase2.run_baseline failed to start. Check the repo/installation.")
if "B2" not in help_result.stdout:
    raise RuntimeError(
        "The GitHub revision cloned above does not include B2 support. "
        "Commit and push the B2 runner changes to the repository's default branch, "
        "then restart this Kaggle session and rerun the notebook. The local project "
        "copy alone is not available to Kaggle."
    )

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN Kaggle secret is empty.")
os.environ["HF_TOKEN"] = hf_token
del hf_token

USE_WANDB = False
WANDB_PROJECT = "phase2-xnli-baselines"
if USE_WANDB:
    wandb_key = secrets.get_secret("WANDB_API_KEY")
    if not wandb_key:
        raise RuntimeError("Enable USE_WANDB only after adding the WANDB_API_KEY Kaggle secret.")
    os.environ["WANDB_API_KEY"] = wandb_key
    del wandb_key

PROTOCOL_PATH = REPO_ROOT / "phase2" / "configs" / "pilot_protocol.json"
ARTIFACT_ROOT = Path("/kaggle/working/phase2_artifacts")
print({"repo": str(REPO_ROOT), "artifact_root": str(ARTIFACT_ROOT), "runner_file": str(runner_path), "wandb": USE_WANDB})


In [ ]:
from phase2.preflight import collect_preflight
from phase2.prepare_data import prepare_xnli
from phase2.protocol import build_manifest, load_protocol
from phase2.run_baseline import validate_prepared_artifacts
import json

preflight = collect_preflight(require_cuda=True, check_gated_model=True)
print(json.dumps(preflight, indent=2, sort_keys=True))

protocol = load_protocol(PROTOCOL_PATH)
if not (ARTIFACT_ROOT / "protocol_manifest.json").is_file():
    if (ARTIFACT_ROOT / "data").exists():
        raise FileExistsError("A partial data directory exists without its manifest. Inspect it before using a fresh artifact root.")
    prepared_manifest = prepare_xnli(PROTOCOL_PATH, ARTIFACT_ROOT)
    print("Prepared fixed XNLI subsets:", prepared_manifest["expected_run_count"], "scheduled runs")
else:
    prepared_manifest = validate_prepared_artifacts(protocol, ARTIFACT_ROOT)
    print("Verified existing fixed data manifest:", prepared_manifest["protocol_sha256"])

ORDER_IDS = [order["id"] for order in protocol["orders"]]
if len(ORDER_IDS) != 5:
    raise ValueError(f"B2 requires all five protocol orders; found {len(ORDER_IDS)}.")
print({"languages": protocol["dataset"]["languages"], "orders_to_run": ORDER_IDS, "buffer_capacity": protocol["replay"]["buffer_capacity_examples"], "data_manifest_sha256": prepared_manifest["data_manifest_sha256"]})


In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    "-m",
    "phase2.run_baseline",
    "--method", "B2",
    "--protocol", str(PROTOCOL_PATH),
    "--artifact-root", str(ARTIFACT_ROOT),
]
for order_id in ORDER_IDS:
    command.extend(["--order-id", order_id])
if USE_WANDB:
    command.extend(["--wandb-project", WANDB_PROJECT])
print("RUN COMMAND:")
print(command)
subprocess.run(command, cwd=REPO_ROOT, check=True)


In [ ]:
from phase2.protocol import build_manifest
from phase2.results import validate_result_record
import statistics
import json

run_manifest = build_manifest(protocol)
run_manifest["data_artifacts"] = prepared_manifest["data_artifacts"]
run_manifest["data_manifest_sha256"] = prepared_manifest["data_manifest_sha256"]
expected_runs = {run["run_id"]: run for run in run_manifest["runs"] if run["method_id"] == "B2"}
if len(expected_runs) != 5:
    raise ValueError(f"Expected five B2 runs in the current manifest; found {len(expected_runs)}.")
completed = {}
for run_id, run in expected_runs.items():
    result_path = ARTIFACT_ROOT / "results" / f"{run_id}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"B2 order is incomplete: {result_path}")
    record = json.loads(result_path.read_text(encoding="utf-8"))
    validate_result_record(record, run_manifest)
    completed[run["order_id"]] = record

metric_names = ("ACC", "BWT", "FWT", "F")
summary = {
    "method_id": "B2",
    "protocol_id": run_manifest["protocol_id"],
    "protocol_sha256": run_manifest["protocol_sha256"],
    "data_manifest_sha256": run_manifest["data_manifest_sha256"],
    "completed_orders": len(completed),
    "expected_orders": len(expected_runs),
    "per_order": {order_id: {name: record["metrics"][name] for name in metric_names} for order_id, record in sorted(completed.items())},
    "across_order_summary": {
        name: {
            "mean": statistics.mean(record["metrics"][name] for record in completed.values()),
            "population_std": statistics.pstdev(record["metrics"][name] for record in completed.values()),
        }
        for name in metric_names
    },
}
summary_path = ARTIFACT_ROOT / "B2_summary.json"
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
temporary_path.replace(summary_path)
print(json.dumps(summary, indent=2, sort_keys=True))
print("Validated all five B2 orders. Summary:", summary_path)


## Saved outputs

- `phase2_artifacts/protocol_manifest.json` and `data/`: frozen selected examples and checksums.
- `phase2_artifacts/checkpoints/<run_id>/`: adapter/tokenizer after each language stage.
- `phase2_artifacts/ledgers/<run_id>/`: complete stage-by-language evaluation matrix and CSV ledger.
- `phase2_artifacts/results/<run_id>.json`: validated Phase 2 result record for aggregation.
- `phase2_artifacts/B2_summary.json`: summary of all five validated orders and across-order metrics.

The B2 `BWT`, `FWT`, and forgetting metrics are computed from the complete fixed-test performance matrix. Replay policy and per-stage exposure are recorded in result metadata. Do not select or tune settings using the test scores.
